# Company Ranking & Top-25 Selection Pipeline

## Overview
This notebook processes scored company data and produces three outputs:
1. **ranked CSV**: All companies ranked by total_score, evidence quality, and revenue
2. **top-25 CSV**: The best 25 companies for immediate outreach
3. **review CSV**: Companies ranked 26-50 for secondary review

## Filtering Logic
- Include companies where `verdict = 'Include'` OR both `E1 = 'Yes'` AND `E2 = 'Yes'`

## Ranking Logic
- Primary: normalized total_score (0-1 scale)
- Secondary: evidence_score (count of non-empty evidence fields)
- Tertiary: revenue_estimate (tie-breaker)

In [3]:
# Setup: imports and file paths
from pathlib import Path
import pandas as pd
import csv
import re

BASE_DIR = Path('/home/uday/Documents/offCampus_placement_assignments/DeepThought/round1')
INPUT_CSV = BASE_DIR / 'output' / 'final_csv' / 'company_scoring_companies.csv'
RANKED_CSV = BASE_DIR / 'output' / 'final_csv' / 'company_scored_companies_ranked.csv'
TOP25_CSV = BASE_DIR / 'output' / 'final_csv' / 'company_top25_shortlist.csv'
REVIEW_CSV = BASE_DIR / 'output' / 'final_csv' / 'company_top25_review.csv'
CLEAN_CSV = BASE_DIR / 'output' / 'final_csv' / 'company_clean_output.csv'

print(f'Input: {INPUT_CSV}')
print(f'Outputs:',)
print(f'  - Ranked: {RANKED_CSV}')
print(f'  - Top-25: {TOP25_CSV}')
print(f'  - Review: {REVIEW_CSV}')

Input: /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_scoring_companies.csv
Outputs:
  - Ranked: /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_scored_companies_ranked.csv
  - Top-25: /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_top25_shortlist.csv
  - Review: /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_top25_review.csv


In [4]:
# Step 1: Load CSV with fallback parser for malformed rows

def load_csv_robust(path):
    """
    Load CSV with multiple fallback strategies:
    1. Try pandas read_csv (fast, handles quotes)
    2. Try csv.reader (standard library, reliable)
    3. Last resort: manual split with heuristic alignment
    """
    try:
        df = pd.read_csv(path, dtype=str)
        print(f'✓ Loaded via pd.read_csv: {df.shape[0]} rows, {df.shape[1]} cols')
        return df
    except Exception as e:
        print(f'✗ pd.read_csv failed: {e}')

    try:
        with open(path, 'r', encoding='utf-8', newline='') as fh:
            reader = csv.reader(fh)
            rows = list(reader)
        if rows:
            header = rows[0]
            data = rows[1:]
            df = pd.DataFrame(data, columns=header, dtype=str)
            print(f'✓ Loaded via csv.reader: {df.shape[0]} rows, {df.shape[1]} cols')
            return df
    except Exception as e:
        print(f'✗ csv.reader failed: {e}')

    # Last resort: manual parsing
    print('Using fallback heuristic parser...')
    header = ['company_name','official_website','C3_score','C3_evidence',
              'C4_score','C4_evidence','C5_score','C5_evidence',
              'C6_score','C6_evidence','C7_score','C7_evidence',
              'C8_score','C8_evidence','total_score','band','verdict',
              'short_personalization_hook','decision_maker','revenue_estimate','primary_sources']
    rows = []
    with open(path, 'r', encoding='utf-8') as fh:
        fh.readline()  # skip header
        for lineno, line in enumerate(fh, start=2):
            tokens = line.rstrip('\n').split(',')
            # If perfect match, use as-is
            if len(tokens) == len(header):
                rows.append(tokens)
                continue
            # Otherwise, right-align: join excess left tokens as company_name
            if len(tokens) > len(header):
                excess = len(tokens) - len(header) + 1
                company_name = ','.join(tokens[:excess])
                rest = tokens[excess:]
                row = [company_name] + rest
            else:
                row = tokens + [''] * (len(header) - len(tokens))
            rows.append(row[:len(header)])
    df = pd.DataFrame(rows, columns=header, dtype=str)
    print(f'✓ Loaded via fallback: {df.shape[0]} rows, {df.shape[1]} cols')
    return df

# Load the data
df = load_csv_robust(INPUT_CSV)
print(f'Columns: {list(df.columns[:5])}...')

✗ pd.read_csv failed: Error tokenizing data. C error: Expected 21 fields in line 80, saw 27

✗ csv.reader failed: 21 columns passed, passed data had 27 columns
Using fallback heuristic parser...
✓ Loaded via fallback: 104 rows, 21 cols
Columns: ['company_name', 'official_website', 'C3_score', 'C3_evidence', 'C4_score']...


In [5]:
# Step 2: Filter companies

# Ensure required columns exist
required_cols = ['verdict', 'E1', 'E2', 'total_score', 'company_name']
for col in required_cols:
    if col not in df.columns:
        df[col] = ''

# Filter: verdict='Include' OR (E1='Yes' AND E2='Yes')
def normalize_str(s):
    return str(s).strip().strip('"').lower() if pd.notna(s) else ''

verdict_include = df['verdict'].apply(normalize_str) == 'include'
e1_yes = df['E1'].apply(normalize_str) == 'yes'
e2_yes = df['E2'].apply(normalize_str) == 'yes'

include_mask = verdict_include | (e1_yes & e2_yes)
df_filtered = df[include_mask].copy()

print(f'Before filter: {len(df)} companies')
print(f'After filter: {len(df_filtered)} companies (Include verdict or both E1=Yes & E2=Yes)')

Before filter: 104 companies
After filter: 77 companies (Include verdict or both E1=Yes & E2=Yes)


In [6]:
# Step 3: Calculate ranking metrics

# Parse total_score as numeric
df_filtered['total_score_num'] = pd.to_numeric(df_filtered['total_score'], errors='coerce').fillna(0)

# Normalize to 0-1
min_score = df_filtered['total_score_num'].min()
max_score = df_filtered['total_score_num'].max()

if max_score > min_score:
    df_filtered['norm_score'] = (df_filtered['total_score_num'] - min_score) / (max_score - min_score)
else:
    df_filtered['norm_score'] = 0.5  # all equal

# Evidence quality: count non-empty evidence fields
evidence_cols = ['C3_evidence', 'C4_evidence', 'C5_evidence', 'C6_evidence', 'C7_evidence', 'C8_evidence']
existing_ev_cols = [c for c in evidence_cols if c in df_filtered.columns]

def count_evidence(row):
    count = 0
    for col in existing_ev_cols:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            count += 1
    return count

df_filtered['evidence_count'] = df_filtered.apply(count_evidence, axis=1)

# Revenue as numeric (for tie-breaking)
def extract_revenue_num(s):
    if pd.isna(s):
        return 0
    # Extract digits from strings like '100 Cr', '100.5 Cr', etc.
    match = re.search(r'([0-9.]+)', str(s))
    return float(match.group(1)) if match else 0

df_filtered['revenue_num'] = df_filtered['revenue_estimate'].apply(extract_revenue_num)

print(f'Norm score range: {df_filtered["norm_score"].min():.2f} to {df_filtered["norm_score"].max():.2f}')
print(f'Evidence count range: {df_filtered["evidence_count"].min()} to {df_filtered["evidence_count"].max()}')
print(f'Revenue range: {df_filtered["revenue_num"].min()} to {df_filtered["revenue_num"].max()}')

Norm score range: 0.00 to 1.00
Evidence count range: 6 to 6
Revenue range: 0.0 to 31724.0


In [7]:
# Step 4: Rank and select top-25

# Sort by: norm_score DESC, evidence_count DESC, revenue_num DESC
df_sorted = df_filtered.sort_values(
    by=['norm_score', 'evidence_count', 'revenue_num'],
    ascending=[False, False, False]
).reset_index(drop=True)

df_sorted['rank'] = range(1, len(df_sorted) + 1)

# Split into top-25 and review (26-50)
top25 = df_sorted[df_sorted['rank'] <= 25]
review = df_sorted[(df_sorted['rank'] > 25) & (df_sorted['rank'] <= 50)]

print(f'Total ranked: {len(df_sorted)} companies')
print(f'Top 25: {len(top25)} companies')
print(f'Review (26-50): {len(review)} companies')
print()
print('Top 5 companies:')
top5_cols = ['rank', 'company_name', 'norm_score', 'evidence_count', 'revenue_num']
print(df_sorted[top5_cols].head(5).to_string(index=False))

Total ranked: 77 companies
Top 25: 25 companies
Review (26-50): 25 companies

Top 5 companies:
 rank                                                                                                                                                                                                            company_name  norm_score  evidence_count  revenue_num
    1                                                                                                                                                                                                Divi's Laboratories Ltd.    1.000000               6      11067.0
    2                                                                                                                                                                                              MSN Laboratories Pvt. Ltd.    0.967742               6       7740.0
    3                                                                                                               

In [8]:
# Step 5: Export CSVs

# Define final output columns (keeping all original + ranking metrics)
output_cols = (
    df_sorted.columns.tolist()  # all columns from original + our computed ones
)

# Export 1: Full ranked
df_sorted[output_cols].to_csv(RANKED_CSV, index=False)
print(f'✓ Exported ranked ({len(df_sorted)} rows): {RANKED_CSV}')

# Export 2: Top-25 only
top25[output_cols].to_csv(TOP25_CSV, index=False)
print(f'✓ Exported top-25 ({len(top25)} rows): {TOP25_CSV}')

# Export 3: Review (26-50)
if len(review) > 0:
    review[output_cols].to_csv(REVIEW_CSV, index=False)
    print(f'✓ Exported review ({len(review)} rows): {REVIEW_CSV}')
else:
    print(f'ℹ No review companies (rank > 25)')

# Export 4: Clean output (original columns only)
original_cols = [c for c in df.columns if c in df_sorted.columns]
df_sorted[original_cols].to_csv(CLEAN_CSV, index=False)
print(f'✓ Exported clean ({len(df_sorted)} rows): {CLEAN_CSV}')

print()
print('✅ Pipeline complete!')

✓ Exported ranked (77 rows): /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_scored_companies_ranked.csv
✓ Exported top-25 (25 rows): /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_top25_shortlist.csv
✓ Exported review (25 rows): /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_top25_review.csv
✓ Exported clean (77 rows): /home/uday/Documents/offCampus_placement_assignments/DeepThought/round1/output/final_csv/company_clean_output.csv

✅ Pipeline complete!
